# V2-2 — đánh giá với báo cáo gốc của bác sĩ
Notebook private. Gắn hai Dataset private: annotations và best_adapter. `smoke` chạy hai ca validation để thử đường ống; `test` chạy đủ 47 ca test bằng protocol đã khóa. Báo cáo sinh ra và bảng rà soát bác sĩ là dữ liệu riêng tư.

In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib
CODE_SHA = "CODE_SHA_PLACEHOLDER"
MODE = "smoke"  # đổi thành "test" sau khi smoke thành công
RUN_NAME = "v2-2-fold1-eval-smoke-01"
REPO = Path("/kaggle/working/repo")
assert MODE in {"smoke", "test"}
assert not REPO.exists()
subprocess.run(["git", "clone", "https://github.com/kttt294/MRI-report-generator.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", CODE_SHA], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-kaggle.txt"), "-r", str(REPO / "requirements-evaluation.txt")], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "Bật GPU T4 trong Kaggle Settings"
protocol = json.loads((REPO / "configs/v2_2_test_protocol.json").read_text(encoding="utf-8"))
candidates = []
for path in Path("/kaggle/input").rglob("adapter_model.safetensors"):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest == protocol["adapter_sha256"]:
        candidates.append(path.parent)
assert len(candidates) == 1, f"Cần đúng một adapter V2-2; tìm được {len(candidates)}"
ADAPTER = candidates[0]
print("GPU:", torch.cuda.get_device_name(0), "Code:", CODE_SHA, "Adapter:", ADAPTER)


In [ ]:
RUN_DIR = Path("/kaggle/working/runs") / RUN_NAME
cmd = [sys.executable, "scripts/evaluate_v2_2.py", "--annotations-root", "/kaggle/input", "--adapter-root", str(ADAPTER), "--protocol", "configs/v2_2_test_protocol.json", "--output", str(RUN_DIR), "--split", "val" if MODE == "smoke" else "test"]
if MODE == "smoke":
    cmd += ["--limit", "2", "--skip-metrics"]
subprocess.run(cmd, cwd=REPO, check=True)
print((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
print("Rà soát lâm sàng:", RUN_DIR / "human_review.csv")
